# HospitalityHub learning notebook: Level 1 and Level 2

This notebook is a learning companion for the HospitalityHub practice project. It records what is built and explains the Angular ideas behind it in beginner-friendly language. Read the Markdown cells from top to bottom; code examples are illustrative and are not executed by this notebook.

**Current status:** Level 1 is substantially complete. Level 2 has started with a global HTTP loading and error interceptor.

Paths in this notebook are relative to the HospitalityHub repository root unless noted otherwise.

## 1. What the application does

HospitalityHub is a small hotel-management app. The Angular client talks to a .NET Web API. The API stores the data in a database through Entity Framework Core.

- **Rooms:** view rooms, add a room, and remove a room. A room has a number, type, nightly price, and availability.
- **Bookings:** select an available room, enter guest details and dates, see an estimated price, save the reservation, and view reservations in the bookings list.
- **Two backend choices:** a SQL Server API and a SQLite API provide the same booking/room endpoints. Run one API at a time and point the Angular `ApiService` at it.
- **Presentation:** responsive room and booking pages share the Poppins / Inter font stack, status messages, and loading/error states.

A saved booking is not just browser state: the client sends it to `POST /api/bookings`, and the .NET API saves it with EF Core. `GET /api/bookings` reads the persisted reservations back.

## 2. Level 1 — CRUD foundations (substantially complete)

CRUD means **Create, Read, Update, Delete**. It describes the basic operations an app performs on stored data.

| Area | Implemented so far |
| --- | --- |
| Rooms | Create a room, read/list rooms, delete a room |
| Bookings | Create a booking, read/list bookings |
| Validation | Required room, guest name, valid email, check-in not in the past, and check-out after check-in |
| Booking rules | The API verifies that the room exists and is available, rejects overlapping stays, and calculates the total from the server's room price |
| Feedback | The UI shows loading, success, validation, and API failure messages |

**Be precise about the word CRUD:** room update is not implemented in the client UI, and booking update/cancellation (delete) are not implemented. The completed operations are useful, but full CRUD for both entities remains a possible follow-up.

### Angular ideas used in Level 1

- **Standalone components** are self-contained UI building blocks; this Angular app does not use feature NgModules.
- **Routes** map browser URLs to screens: `/` for rooms, `/book` for the reservation form, and `/bookings` for the booking register.
- **Dependency injection** lets a component receive the shared `ApiService` rather than constructing HTTP code itself.
- **`HttpClient` and Observables:** calling the API returns an Observable. The app subscribes to the result to update UI state when a response succeeds or fails.
- **Reactive forms:** the form model stores values and validation state separately from the HTML. Validators keep bad or incomplete data from being submitted.
- **Signals:** signals hold UI state such as rooms, loading, and errors. A template reading a signal updates when that signal changes.
- **Backend validation is still required:** browser validation improves usability, but the API must enforce important booking rules because clients can be bypassed.

### Where to explore the Level 1 code

- `hospitality-client/src/app/app.routes.ts` — screen routing.
- `hospitality-client/src/app/core/services/api.ts` — the HTTP service used by features.
- `hospitality-client/src/app/core/models/room.ts` and `booking.ts` — TypeScript data shapes.
- `hospitality-client/src/app/features/rooms/room-list/` — room management UI.
- `hospitality-client/src/app/features/bookings/booking-form/` — validated create-booking flow.
- `hospitality-client/src/app/features/bookings/booking-list/` — persisted booking register.
- `HospitalityHub.Api/Controllers/BookingsController.cs` and `HospitalityHub.Api.Sqlite/Controllers/BookingsController.cs` — booking endpoints for each database provider.

## 3. Level 2 — What is an HTTP interceptor?

Imagine every outgoing parcel passes through a reception desk. The desk can stamp it, record it, or notice that delivery failed. An HTTP interceptor is that reception desk for Angular HTTP requests and responses.

Without an interceptor, every feature repeats the same loading and error plumbing. With an interceptor, one shared function can observe requests made by `HttpClient` and apply cross-cutting behavior consistently.

The simplified flow is:

1. A component calls `ApiService.getRooms()`.
2. `ApiService` calls `HttpClient`.
3. Angular runs the configured interceptor before sending the HTTP request.
4. The response (or HTTP error) travels back through the interceptor.
5. The subscriber in the feature receives the response or the same error.

Interceptors only see requests made through Angular `HttpClient`; they do not intercept browser navigation or arbitrary `fetch()` calls.

### Modern Angular: functional interceptors

This project uses `HttpInterceptorFn`, Angular's functional interceptor type. `withInterceptors([...])` registers it with `provideHttpClient(...)` in `app.config.ts`. Functional interceptors can use Angular dependency injection via `inject()` while the interceptor is running.

Official Angular guide: [HTTP interceptors](https://v21.angular.dev/guide/http/interceptors).

## 4. Loading indicator: why we count requests

Two API calls may happen at the same time (for example, the room list and another screen's data). A boolean is not enough: if request A finishes first, setting `loading = false` would hide the indicator while request B is still running.

Instead, `GlobalLoadingService` holds an **active request count**:

1. When an HTTP request begins, increment the count.
2. When it completes, errors, or is cancelled, decrement it in RxJS `finalize()`.
3. The global indicator is visible exactly while the count is greater than zero.

`finalize()` is important because it runs for success, error, and unsubscribe/cancellation. The service clamps the count at zero as a defensive guard.

The app uses a slim progress line and a small spinner/status pill. The status region and visually hidden text allow assistive technology to announce that data is loading. Because the indicator is global, feature components can still keep their more specific loading or empty states.

## 5. Global error feedback without swallowing errors

The interceptor detects `HttpErrorResponse` and adds a readable message to `GlobalApiErrorService`. The root `GlobalHttpStatusComponent` displays dismissible notifications, so errors from any `HttpClient` call can be seen even when the current page has no dedicated error panel.

**The interceptor rethrows the original error.** `throwError(() => error)` passes it back to the caller instead of pretending the request succeeded. The booking and room screens can therefore still show context-specific inline errors. A global toast and an inline error may both appear for the same failed operation: the toast is the app-wide alert; the inline message explains what failed in that particular form or list.

Typical cases include:

- `status === 0`: the browser could not connect (API offline, certificate, or CORS issue).
- A server response such as 400, 404, or 500: show a safe API problem title/detail when present, otherwise show the HTTP status.

The global notification service keeps a small recent queue and supports a keyboard-accessible dismiss button. Notifications use alert/live-region semantics. Do not put credentials or sensitive request payloads into global error messages or logs.

### Optional next refinement: opt out per request

Some calls may intentionally be quiet, such as background refreshes. Angular supports `HttpContextToken` so an individual request can pass metadata that the interceptor reads. This project does not currently opt out any requests; add that only when a real quiet-request requirement exists.

## 6. Walk through the implementation

### Registration — `hospitality-client/src/app/app.config.ts`

The app calls `provideHttpClient(withInterceptors([loadingErrorInterceptor]))`. If this registration is missing, Angular never runs the interceptor. `provideHttpClient()` remains the provider that makes `HttpClient` available.

### Request/response middleware — `hospitality-client/src/app/core/interceptors/loading-error.interceptor.ts`

The interceptor injects the two shared services, marks the request as active, and returns the Observable from `next(request)`. `catchError` publishes an `HttpErrorResponse` message and rethrows the same error. `finalize` always tells the loading service that this request is finished.

The request object is immutable. If an interceptor needs to change headers or URL, it must create a copy using `request.clone(...)`; this loading/error interceptor does not need to change the request.

### Shared state — `hospitality-client/src/app/core/services/global-loading.ts`

`activeRequestCount` is the source of truth. `isLoading` is a computed signal derived from the count. The interceptor changes the count; the UI reads the computed signal.

### Error queue — `hospitality-client/src/app/core/services/global-api-error.ts`

This service stores recent `{ id, message }` notifications in a signal and removes an item when the user dismisses it.

### Global UI — `hospitality-client/src/app/core/components/global-http-status.ts`

The component reads loading and error signals and renders the progress bar, spinner pill, and notification stack. It is included in the root `app.html` beside `<router-outlet>`, so it is present on every route.

### Tests — `hospitality-client/src/app/core/interceptors/loading-error.interceptor.spec.ts`

The HTTP testing backend verifies concurrent requests keep the spinner active until the last request completes. It also verifies an error creates a global message, remains the same error for the subscriber, and ends the loading state.

## 7. Run and verify the project

Open a terminal in `hospitality-client` and run:

```powershell
npm install
npm start
npm test -- --watch=false
npm run build
```

Start either `HospitalityHub.Api` (SQL Server) or `HospitalityHub.Api.Sqlite` (SQLite) in another terminal. Confirm the selected `ApiService` base URL, that the backend is running, and that its CORS policy permits the exact Angular origin. A browser CORS error can look like a network error even when the API process is running.

### Interceptor learning checklist

- [ ] I can explain what request middleware does and why an app might use it.
- [ ] I can trace a call from component -> `ApiService` -> `HttpClient` -> interceptor -> backend -> subscriber.
- [ ] I understand why active requests need a counter instead of one boolean.
- [ ] I can explain why `finalize()` clears loading state on success, failure, and cancellation.
- [ ] I can find where `withInterceptors` registers the function.
- [ ] I know that `catchError` must rethrow when the feature still needs to handle the failure.
- [ ] I can run the interceptor tests and explain what each assertion proves.

### Try these exercises

1. Temporarily stop the API and load the Rooms page. Observe the global error notification and the screen's inline error. Start the API again and retry.
2. In the interceptor test, start two requests. Complete one, verify loading remains true, then complete the second and verify loading becomes false.
3. Add a request timeout or retry policy only after deciding how it should interact with the global loading count and error notifications.
4. As a future exercise, add an `HttpContextToken` to suppress a global toast for a deliberate background request, while still rethrowing the error to the caller.

### Common mistakes and how to diagnose them

| Symptom | Check |
| --- | --- |
| No global loading bar or error toast | Ensure the interceptor is passed to `withInterceptors` in `app.config.ts`, and the global status component is rendered by the root app. |
| Loading indicator gets stuck | Ensure every request stream reaches `finalize`; avoid returning a separate Observable that bypasses the original pipeline. |
| Indicator disappears while another call is active | Use an active-request counter, not a shared boolean set to false by each request. |
| Feature no longer handles an HTTP error | Do not return a success-shaped fallback from `catchError`; rethrow the original error. |
| Browser says CORS or status 0 | Check the running API, trusted HTTPS certificate, configured API URL, and exact allowed frontend origin. |
| Duplicate-looking error messages | The global notification is intentional and the page may also show contextual inline feedback. If the UX becomes noisy, design a per-request opt-out with `HttpContextToken`. |

## 8. Roadmap after this notebook

Level 2 has started with interceptors. Possible next topics are route guards, lazy-loaded feature routes, stronger automated tests, booking cancellation/update, and deployment configuration. Choose one topic at a time and add a test or a small exercise to make the concept stick.